# COMSE6998-007 — Introduction to LLM Based Generative AI Systems
## Fall 2026 · Homework 2 · Problem 2 — Contrastive Decoding

**Columbia University**

**Instructors:** Parijat Dube (pd2637) and Chen Wang (cw3687)  
**TAs:** Mayur Kulkarni (msk2277), Himanshu Jhawar (hj2713), Aman Upganlawar (au2327)  
**Released:** Oct. 2, 2026 · **Due:** Oct. 16, 2026  
**Points:** 25

---

Implement **expert–amateur contrastive decoding** from [Li et al. (2022)](https://arxiv.org/abs/2210.15097),
not Hugging Face's hidden-state-based *contrastive search*. This independently authored
implementation uses **stock Transformers 4.57** and a compatible preinstalled PyTorch
(GPU stack preserved; originally tested with 2.10), not the upstream fork. All graded functions
live in this notebook. The visible ungraded helper-export cell below supplies reusable
model/data loading, metrics, persistence, and presentation infrastructure. Upload **only this
notebook** to Colab using **File → Upload notebook**; no companion files are required to *run* it.
Cloning and reading the reference repository is nevertheless a **graded setup task (2a)**:
record your clone's commit SHA in the setup documentation cell of Part 2.

| Part / rubric item | Points |
|---|---:|
| 1. Intuition and comparison | 2 |
| 1. Objective, false positives and false negatives | 2 |
| 1. Adaptive plausibility and alpha | 2 |
| 2. Reference clone, environment, WikiText preprocessing | 2 |
| 2. Smaller same-family amateur and ablation implementation | 3 |
| 3. Nine configurations; diversity, MAUVE, coherence-proxy table | 8 |
| 3. Metric trade-offs and relation to paper | 3 |
| 3. Two–three matched best/worst examples and commentary | 3 |
| **Total: concepts 6 + setup/implementation 5 + evaluation 14** | **25** |

**Safe defaults:** model/data downloads and experiments are opt-in; package installation runs
only when you execute its setup cell. There are no stored benchmark claims or paid API calls.
Select **Runtime → Change runtime type → GPU** if available, then run setup in order.
The notebook does not need to exist in the runtime filesystem and never changes your cwd.

**Student starter: complete genuine TODO stubs below; no hidden reference implementation.**


## Part 1 — Conceptual understanding (6 points)
1. **(2 points)** Explain the core intuition behind contrastive decoding. How does it differ from
   standard decoding strategies like greedy decoding, beam search, or nucleus sampling? What problem
   is contrastive decoding trying to solve?
2. **(2 points)** Describe the mathematical formulation of the contrastive objective. Why may
   unconstrained maximization of the contrastive objective result in false negatives and false
   positives during generation?
3. **(2 points)** Why do we need an adaptive plausibility constraint in the full contrastive decoding
   formulation? How does the choice of α affect the generated text?


### Your conceptual answers
1. Contrastive decoding uses a larger expert model and a smaller amateur model. It compares their next-token probabilities and favors tokens that the expert supports more strongly relative to the amateur. An expert-based plausibility constraint keeps unlikely tokens out of consideration.

Greedy decoding picks the highest-probability token at each step. Beam search keeps a fixed number of high-scoring partial sequences and expands them step by step. Nucleus sampling randomly selects a token from the smallest set whose cumulative probability reaches a threshold.

These methods usually rely on one model. Contrastive decoding uses the amateur to help identify patterns such as repetition and generic text that a smaller model also tends to produce. The goal is to improve generation quality while keeping the output plausible under the expert.

2. The contrastive objective is to maximize the difference between the expert and amateur log probabilities:

score = log(expert probability) − log(amateur probability)

For a generated sequence, the objective sums these scores over its tokens.

Without a plausibility constraint, an unlikely token can get a high score if its amateur probability is even lower. This creates a false positive. A good token can also receive a low score when both models assign it high probability. This creates a false negative because agreement from the amateur can penalize an otherwise appropriate choice.

3. The adaptive plausibility constraint restricts contrastive decoding to tokens that the expert considers sufficiently likely. It prevents unlikely tokens from winning simply because their amateur probabilities are even lower. When the expert is highly confident, it also reduces the risk of rejecting a good token because both models agree on it.

The threshold is alpha times the highest expert probability at the current step, so it adapts to the expert's predictions. A higher alpha keeps fewer candidates and makes generation more conservative. A lower alpha allows more candidates to compete using the contrastive score, potentially increasing diversity but also risking less plausible output.


## Part 2 — Setup and implementation (5 points)
### 2a. Reference checkout, environment, and preprocessing (2 points)
The handout asks for cloning the reference repository. Do this **for reading only**, outside the
submission directory, and record the actual commit returned by the commands below in `UPSTREAM_COMMIT`.
No revision has been invented or claimed as already cloned by this notebook.

```bash
git clone https://github.com/XiangLi1999/ContrastiveDecoding.git contrastive-decoding-reference
git -C contrastive-decoding-reference rev-parse HEAD
# Inspect README and equations; do NOT pip-install its transformers fork or setup.py.
```

**Colab setup (ungraded infrastructure):** run the workspace, core install, helper export,
and imports cells in that order. The workspace is temporary, absolute, and isolated by notebook
variant; rerunning setup in the same kernel reuses it. No `.py` or requirements upload is needed.
The core install preserves existing Torch/torchvision/torchaudio wheels and binary packages.
If pip reports a constraint conflict, do not force an upgrade: use a fresh compatible runtime.
If a loaded package changes, **stop and restart the runtime/session**, then rerun setup from
the top. Changing runtime type resets the environment. Local users may use their own virtual
environment; repository companion files are maintenance sources, not execution requirements.

The separate **opt-in MAUVE install** is unnecessary for pilots but required for full evaluation;
missing MAUVE, FAISS, or scikit-learn means an explicit **incomplete run**, never a proxy score.
Record the commit SHA of your clone (the `rev-parse HEAD` output above): set `UPSTREAM_COMMIT`
in the imports cell to that 40-hexadecimal SHA and report it in the setup documentation cell
below; this is part of graded Task 2a. The code never executes the clone, so the notebook still
runs with `None`, but leaving the SHA unrecorded leaves Task 2a incomplete.
Report versions/hardware from `environment_metadata()`. Defaults are float32 GPT-2 medium
(expert) and GPT-2 small (amateur); use CUDA if available and record the device. Model sizes are
configurable, but require a smaller same-family amateur, identical token-ID mapping **and BPE
pipeline/special IDs**, and at least 1024 positions for the prescribed ablation.

**Memory/time warning:** the full 100-window × nine-condition uncached generation takes roughly
7–8 hours on a Colab T4 GPU. Generation, scoring and MAUVE feature extraction may exceed Colab
VRAM/RAM or session limits, so start early and save results as you go. GPU availability is not
guaranteed. `MAUVE_DEVICE_ID = -1` defaults to CPU to avoid adding another GPU model; set it to
an available GPU index only if sufficient memory remains. CPU MAUVE still consumes RAM/time.
Keep pilots labeled incomplete, and download results before the ephemeral runtime is deleted.

**Data protocol:** Hugging Face `Salesforce/wikitext`, `wikitext-103-raw-v1`, `test` split,
corresponding to the handout's [WikiText-103 source](https://academictorrents.com/details/a4fee5547056c845e31ab952598f43b42333183c).
Top-level `= Title =` lines begin articles; retain inner headings but exclude the article title.
Join article body rows with newlines; tokenize each article without added special tokens. Take
complete **768-token prompt + 128-token human continuation** candidates, stride **128**,
never crossing an article boundary. Prompts overlap within an article; reference spans do not.
Overlapping prompts are statistically dependent: discuss this and use article-level resampling
if estimating uncertainty. Stride 896 would leave too few test-split candidates for large MAUVE
runs. Seeded reservoir sampling selects the requested count uniformly from candidates; sort the
selected windows into source order. Preserve article indices/titles/token offsets, raw texts and
exact IDs. All nine conditions reuse exactly these windows.

The full sample count is **100** windows, kept small so the evaluation fits a reasonable GPU budget;
larger counts are optional, subject to available complete article windows. MAUVE from a few dozen samples is unstable. A separate
**64-token-prompt, 32-token-generation, 3-window pilot** (stride 32) checks plumbing only: its prompt
does not exceed 512, so it cannot demonstrate the full-vs-half-window effect and is NOT the benchmark.
Hub revisions are resolved to immutable SHAs before model/data downloads and saved with dataset
fingerprint, seeds, versions and parameters. Reuse these SHAs for later reruns rather than `main`.


In [ ]:
!git clone https://github.com/XiangLi1999/ContrastiveDecoding.git contrastive-decoding-reference
!git -C contrastive-decoding-reference rev-parse HEAD

Cloning into 'contrastive-decoding-reference'...
remote: Enumerating objects: 1648, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 1648 (delta 2), reused 12 (delta 1), pack-reused 1630 (from 1)
Receiving objects: 100% (1648/1648), 259.49 MiB | 14.56 MiB/s, done.
Resolving deltas: 100% (489/489), done.
Updating files: 100% (1349/1349), done.
170e9142e92159c1237d731e240f5eb14aabf428


### Your setup documentation (2 pts — Task 2a)

TODO: record (1) the commit SHA of your clone of the ContrastiveDecoding repository, (2) the packages and versions you used (see `environment_metadata()`), and (3) the preprocessing steps applied to WikiText-103.


In [ ]:
# Standard-library only; no uploads, repository discovery, or cwd changes.
from pathlib import Path
import sys
import tempfile

_P2_VARIANT = "student"
if not hasattr(sys, "_hw2_p2_workspaces"):
    sys._hw2_p2_workspaces = {}
if _P2_VARIANT not in sys._hw2_p2_workspaces:
    sys._hw2_p2_workspaces[_P2_VARIANT] = tempfile.mkdtemp(prefix=f"hw2-p2-{_P2_VARIANT}-")
P2_WORKSPACE = Path(sys._hw2_p2_workspaces[_P2_VARIANT]).resolve()
P2_WORKSPACE.mkdir(parents=True, exist_ok=True)
P2_HELPER_PATH = P2_WORKSPACE / "contrastive_utils.py"
P2_RESULTS_DIR = P2_WORKSPACE / "results"
print("P2 workspace:", P2_WORKSPACE)


P2 workspace: /tmp/hw2-p2-student-9kqrawis


In [ ]:
# Run before importing ML libraries. No external requirements file is needed.
import importlib.metadata as _metadata
import re
import subprocess
import sys

def _p2_versions():
    return {re.sub(r"[-_.]+", "-", d.metadata["Name"]).lower(): d.version
            for d in _metadata.distributions() if d.metadata["Name"]}

def _p2_install(requirements):
    if getattr(sys, "_hw2_p2_restart_required", False):
        raise RuntimeError("Restart the runtime/session, then rerun setup from the top.")
    before = _p2_versions()
    version = before.get("torch", "")
    match = re.match(r"(\d+)\.(\d+)", version)
    if not match or tuple(map(int, match.groups())) < (2, 4):
        raise RuntimeError("Use a fresh Colab runtime with PyTorch >=2.4 (GPU recommended). "
                           "This cell deliberately does not replace your Torch/CUDA stack.")
    loaded = {name.split(".")[0] for name in sys.modules}
    loaded_distributions = {
        re.sub(r"[-_.]+", "-", dist).lower()
        for module, distributions in _metadata.packages_distributions().items()
        if module in loaded for dist in distributions
    }
    # Keep preinstalled GPU wheels and binary ABI dependencies intact, even before imports.
    protected = loaded_distributions | {
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "pandas", "pyarrow",
        "scikit-learn", "matplotlib", "pillow", "faiss-cpu",
    }
    constraints = P2_WORKSPACE / "installed-stack-constraints.txt"
    constraints.write_text("".join(f"{name}=={before[name]}\n" for name in sorted(protected)
                                   if name in before), encoding="utf-8")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "--constraint", str(constraints),
                        *requirements], check=True)
    finally:
        after = _p2_versions()
        changed = sorted(name for name in loaded_distributions
                         if before.get(name) != after.get(name))
        if changed:
            sys._hw2_p2_restart_required = True
            raise RuntimeError(f"Loaded packages changed: {changed}. STOP: restart the runtime/session "
                               "and rerun all setup cells before importing or running models.")
    print("Dependencies ready; protected the existing Torch/GPU and binary stack.")

# Resolver conflicts are intentional safety stops, not permission to force upgrades.
# Use a fresh runtime if its preinstalled stack cannot satisfy these requirements.
_p2_install(["transformers>=4.57,<4.58", "datasets>=3,<5", "pandas>=2,<3", "matplotlib>=3.8,<4"])


Dependencies ready; protected the existing Torch/GPU and binary stack.


In [ ]:
# Opt in BEFORE the imports cell when planning a genuine full evaluation.
INSTALL_MAUVE = True
if INSTALL_MAUVE:
    _p2_install(["mauve-text>=0.4,<0.5", "faiss-cpu>=1.8,<2", "scikit-learn>=1.4,<2"])
else:
    print("MAUVE installation skipped; pilots remain explicitly incomplete.")


Dependencies ready; protected the existing Torch/GPU and binary stack.


In [ ]:
%%writefile "{P2_HELPER_PATH}"
"""Ungraded HW2 infrastructure; no decoding, diversity, or experiment solutions.

This file is independently distributable with the student notebook. Network/model
loading is explicit: importing the module never downloads or starts a benchmark.
"""
from __future__ import annotations
import os
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import importlib.metadata
import json
import math
import operator
from pathlib import Path
import platform
import random
import re
import sys

import torch
import torch.nn.functional as F


def context_limit(model):
    """GPT-2-style learned absolute-position capacity."""
    for name in ("max_position_embeddings", "n_positions"):
        value = getattr(model.config, name, None)
        if value is not None:
            return int(value)
    raise ValueError("Model must expose its absolute-position context capacity")


def model_device(model):
    return next(model.parameters()).device


def token_ids(ids, vocab_size):
    """Validate without silently truncating floating-point token IDs."""
    result = []
    for value in ids:
        if isinstance(value, bool):
            raise ValueError("Boolean token IDs are invalid")
        try:
            value = operator.index(value)
        except TypeError as exc:
            raise ValueError("Token IDs must be integers") from exc
        if not 0 <= value < vocab_size:
            raise ValueError("Token ID outside model vocabulary")
        result.append(value)
    return result


def validate_tokenizers(expert_tokenizer, amateur_tokenizer):
    """Require identical token-ID mapping, special IDs, and tokenization pipeline.

    Both loaders request fast tokenizers. Comparing the backend model includes
    BPE merges, not just the vocabulary; equal vocabularies alone are insufficient.
    Padding/truncation runtime settings are deliberately not part of this check.
    """
    if expert_tokenizer.get_vocab() != amateur_tokenizer.get_vocab():
        raise ValueError("Expert/amateur token-to-ID mappings differ")
    for key in ("bos_token_id", "eos_token_id", "unk_token_id", "pad_token_id"):
        if getattr(expert_tokenizer, key) != getattr(amateur_tokenizer, key):
            raise ValueError(f"Tokenizer special ID mismatch: {key}")
    if not all(getattr(t, "is_fast", False) for t in (expert_tokenizer, amateur_tokenizer)):
        raise ValueError("Fast tokenizers required to verify the BPE pipeline")
    left = json.loads(expert_tokenizer.backend_tokenizer.to_str())
    right = json.loads(amateur_tokenizer.backend_tokenizer.to_str())
    for key in ("model", "normalizer", "pre_tokenizer", "post_processor", "decoder", "added_tokens"):
        if left.get(key) != right.get(key):
            raise ValueError(f"Tokenizer pipeline mismatch: {key}")
    return True


def environment_metadata():
    packages = {}
    for package in ("torch", "transformers", "datasets", "numpy", "mauve-text", "faiss-cpu", "scikit-learn"):
        try:
            packages[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            packages[package] = None
    return {
        "python": sys.version, "platform": platform.platform(), "packages": packages,
        "cuda_available": torch.cuda.is_available(),
        "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "cpu": platform.processor(),
    }


def load_model_pair(expert_id="openai-community/gpt2-medium", amateur_id="openai-community/gpt2",
                    expert_revision="main", amateur_revision="main", device="cpu"):
    """Explicit online opt-in; resolve mutable revisions to immutable Hub commits."""
    from huggingface_hub import HfApi
    from transformers import AutoModelForCausalLM, AutoTokenizer

    api = HfApi()
    expert_sha = api.model_info(expert_id, revision=expert_revision).sha
    amateur_sha = api.model_info(amateur_id, revision=amateur_revision).sha
    et = AutoTokenizer.from_pretrained(expert_id, revision=expert_sha, use_fast=True)
    at = AutoTokenizer.from_pretrained(amateur_id, revision=amateur_sha, use_fast=True)
    validate_tokenizers(et, at)
    expert = AutoModelForCausalLM.from_pretrained(expert_id, revision=expert_sha).to(device).eval()
    amateur = AutoModelForCausalLM.from_pretrained(amateur_id, revision=amateur_sha).to(device).eval()
    if any(m.config.model_type != "gpt2" for m in (expert, amateur)):
        raise ValueError("This position-handling protocol requires two GPT-2-family models")
    if any(m.get_output_embeddings().weight.shape[0] != len(et) for m in (expert, amateur)):
        raise ValueError("Model output vocabulary must exactly match tokenizer")
    expert_parameters = sum(p.numel() for p in expert.parameters())
    amateur_parameters = sum(p.numel() for p in amateur.parameters())
    if amateur_parameters >= expert_parameters:
        raise ValueError("Choose a strictly smaller amateur model than the expert")
    if min(context_limit(expert), context_limit(amateur)) < 1024:
        raise ValueError("Both models must support the prescribed 1024-position grid")
    metadata = {
        "expert": {"id": expert_id, "revision": expert_sha, "parameters": expert_parameters,
                   "context_limit": context_limit(expert)},
        "amateur": {"id": amateur_id, "revision": amateur_sha, "parameters": amateur_parameters,
                    "context_limit": context_limit(amateur)},
        "tokenizer_verified": True, "device": str(device), "dtype": str(next(expert.parameters()).dtype),
    }
    return expert, amateur, et, metadata


def article_windows(rows, tokenizer, *, count, prompt_tokens=768, reference_tokens=128, seed=17, stride=None):
    """Sample windows within WikiText top-level articles.

    A top-level '= Title =' row starts an article; '= = Section = =' does not.
    Discard pre-heading material, exclude the article title, retain section headings
    and join body rows with newlines. Tokenize each article once, with no inserted
    BOS/EOS. Default stride equals reference length: prompts overlap, but reference
    spans do not. Report this within-article dependence. Never bridge articles.
    Reservoir sampling is uniform over candidates and bounds token-window memory.
    """
    stride = reference_tokens if stride is None else stride
    for value in (count, prompt_tokens, reference_tokens, stride):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError("Window counts, lengths and stride must be positive integers")
    rng = random.Random(seed)
    width = prompt_tokens + reference_tokens
    reservoir, seen, article_index = [], 0, -1
    title, lines = None, []

    def consume(article_title, body, index):
        nonlocal seen
        if article_title is None:
            return
        ids = tokenizer.encode("\n".join(body), add_special_tokens=False)
        for offset in range(0, len(ids) - width + 1, stride):
            seen += 1
            slot = seen - 1 if seen <= count else rng.randrange(seen)
            if slot >= count:
                continue
            prompt = ids[offset:offset + prompt_tokens]
            reference = ids[offset + prompt_tokens:offset + width]
            item = {
                "window_id": f"article-{index}:token-{offset}", "article_index": index,
                "article_title": article_title, "article_token_offset": offset,
                "article_token_count": len(ids), "prompt_ids": prompt, "reference_ids": reference,
                "prompt_text": tokenizer.decode(prompt, clean_up_tokenization_spaces=False),
                "reference_text": tokenizer.decode(reference, clean_up_tokenization_spaces=False),
            }
            if seen <= count:
                reservoir.append(item)
            else:
                reservoir[slot] = item

    for row in rows:
        text = row["text"]
        stripped = text.strip()
        is_title = bool(re.fullmatch(r"=\s+[^=].*?\s+=", stripped)) and "=" not in stripped[1:-1]
        if is_title:
            consume(title, lines, article_index)
            article_index += 1
            title, lines = stripped[1:-1].strip(), []
        elif title is not None:
            lines.append(text.rstrip("\n"))
    consume(title, lines, article_index)
    if seen < count:
        raise ValueError(f"Only {seen} complete article windows available, requested {count}")
    reservoir.sort(key=lambda w: (w["article_index"], w["article_token_offset"]))
    return reservoir


def load_wikitext_windows(tokenizer, *, count=500, prompt_tokens=768, reference_tokens=128,
                          seed=17, revision="main"):
    from datasets import load_dataset
    from huggingface_hub import HfApi

    dataset_id = "Salesforce/wikitext"
    sha = HfApi().dataset_info(dataset_id, revision=revision).sha
    dataset = load_dataset(dataset_id, "wikitext-103-raw-v1", split="test", revision=sha)
    windows = article_windows(dataset, tokenizer, count=count, prompt_tokens=prompt_tokens,
                              reference_tokens=reference_tokens, seed=seed)
    metadata = {
        "id": dataset_id, "config": "wikitext-103-raw-v1", "split": "test", "revision": sha,
        "fingerprint": dataset._fingerprint, "seed": seed, "sample_count": count,
        "prompt_tokens": prompt_tokens, "reference_tokens": reference_tokens,
        "stride": reference_tokens, "sampling": "uniform reservoir, sorted in source order",
        "window_dependence": "overlapping prompts within articles; nonoverlapping reference spans",
        "article_boundary_policy": "top-level headings; no cross-article windows; title excluded",
    }
    return windows, metadata


@torch.inference_mode()
def conditional_perplexity(model, prompts, continuations, *, pad_token_id, batch_size=8):
    """Token-weighted conditional NLL of continuation tokens only.

    Right-pad prompt+continuation and mask all prompt and padding labels to -100.
    The shifted target at the first continuation position is predicted by the
    final prompt position. EOS is excluded if absent from the supplied continuation.
    Empty continuations contribute zero targets, not a spurious zero-loss sample.
    Inputs must fit the scorer context; silently truncating would change the proxy.
    """
    if len(prompts) != len(continuations) or batch_size <= 0:
        raise ValueError("Matched prompts/continuations and positive batch_size required")
    vocab = model.get_output_embeddings().weight.shape[0]
    pad_token_id = token_ids([pad_token_id], vocab)[0]
    pairs = []
    for prompt, continuation in zip(prompts, continuations):
        p, c = token_ids(prompt, vocab), token_ids(continuation, vocab)
        if not p:
            raise ValueError("Each continuation needs a nonempty conditioning prompt")
        if len(p) + len(c) > context_limit(model):
            raise ValueError("Conditional sequence exceeds scorer position capacity")
        pairs.append((p, c))
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for start in range(0, len(pairs), batch_size):
        batch = [(p, c) for p, c in pairs[start:start + batch_size] if c]
        if not batch:
            continue
        width = max(len(p) + len(c) for p, c in batch)
        ids = torch.full((len(batch), width), pad_token_id, dtype=torch.long, device=model_device(model))
        attention = torch.zeros_like(ids)
        labels = torch.full_like(ids, -100)
        for i, (p, c) in enumerate(batch):
            length = len(p) + len(c)
            ids[i, :length] = torch.tensor(p + c, device=ids.device)
            attention[i, :length] = 1
            labels[i, len(p):length] = torch.tensor(c, device=ids.device)
        positions = torch.arange(width, device=ids.device).unsqueeze(0).expand_as(ids)
        logits = model(input_ids=ids, attention_mask=attention, position_ids=positions,
                       use_cache=False).logits.float()
        targets = labels[:, 1:]
        nll = F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]), targets.reshape(-1),
                              ignore_index=-100, reduction="sum")
        if not torch.isfinite(nll):
            raise ValueError("Nonfinite continuation NLL")
        total_nll += nll.item()
        total_tokens += (targets != -100).sum().item()
    mean_nll = total_nll / total_tokens if total_tokens else None
    perplexity = math.exp(mean_nll) if mean_nll is not None and mean_nll < 709 else None
    return {"nll_sum": total_nll, "scored_tokens": total_tokens, "mean_nll": mean_nll,
            "perplexity": perplexity, "empty_continuations": sum(not c for _, c in pairs)}


def require_mauve():
    """Fail explicitly: full evaluation may never substitute a proxy for MAUVE."""
    try:
        import mauve
        import faiss  # noqa: F401
        from sklearn.cluster import KMeans  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("INCOMPLETE: genuine MAUVE dependencies missing. Run the notebook's optional "
                           "MAUVE install cell; pilot results are not full evaluation.") from exc
    return mauve


def prepare_mauve(*, feature_model="openai-community/gpt2", revision="main", seed=17, device_id=-1):
    """Explicit download, pinned feature-model snapshot; fixed clustering settings."""
    require_mauve()
    from huggingface_hub import HfApi, snapshot_download

    sha = HfApi().model_info(feature_model, revision=revision).sha
    local_path = snapshot_download(feature_model, revision=sha,
                                   allow_patterns=["*.json", "*.txt", "*.safetensors"])
    return {"feature_model": feature_model, "feature_revision": sha,
            "featurize_model_name": local_path, "device_id": device_id,
            "max_text_length": 128, "num_buckets": 25, "seed": seed,
            "kmeans_num_redo": 5, "kmeans_max_iter": 500, "mauve_scaling_factor": 5,
            "batch_size": 8}


def genuine_mauve(references, generations, settings):
    mauve = require_mauve()
    if len(references) != len(generations) or len(references) < settings["num_buckets"]:
        raise ValueError("MAUVE needs matched samples and at least num_buckets samples")
    if any(not text.strip() for text in references + generations):
        raise ValueError("INCOMPLETE: empty text cannot be silently removed from matched MAUVE inputs")
    keys = ("featurize_model_name", "device_id", "max_text_length", "num_buckets", "seed",
            "kmeans_num_redo", "kmeans_max_iter", "mauve_scaling_factor", "batch_size")
    result = mauve.compute_mauve(p_text=references, q_text=generations,
                                 **{key: settings[key] for key in keys}, verbose=False)
    value = float(result.mauve)
    if not math.isfinite(value) or not 0 <= value <= 1:
        raise ValueError("Invalid MAUVE result")
    return value


def save_results(path, result):
    """Persist exact token IDs, raw examples, metrics, settings and revisions."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text + "\n", encoding="utf-8")
    temporary.replace(path)


def plot_diversity(result):
    """Measured-only monochrome small multiples; the notebook also shows a table."""
    import matplotlib.pyplot as plt

    rows = result["rows"]
    if len(rows) != 9:
        raise ValueError("Plot requires all nine measured configurations")
    with plt.rc_context({"figure.facecolor": "white", "axes.facecolor": "white",
                         "text.color": "#222222", "axes.labelcolor": "#222222",
                         "xtick.color": "#222222", "ytick.color": "#222222"}):
        fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), sharey=True, layout="constrained")
        for ax, window, marker in zip(axes, (1024, 512, 1), ("o", "s", "^")):
            group = sorted((r for r in rows if r["amateur_window"] == window),
                           key=lambda r: r["temperature"])
            ax.plot([r["temperature"] for r in group], [r["diversity"] for r in group],
                    color="#222222", linewidth=1.5, marker=marker, markersize=7,
                    markeredgecolor="white", markeredgewidth=1.5)
            ax.set(title=f"Amateur window: {window}", xlabel="Amateur temperature",
                   ylim=(0, 1.05), xticks=(0.5, 1.0, 1.5))
            ax.grid(axis="y", color="#e5e5e5", linewidth=0.6)
            ax.set_axisbelow(True)
            for spine in ax.spines.values():
                spine.set_color("#dddddd")
        axes[0].set_ylabel("Mean distinct-2/3/4 (higher is more diverse)")
        label = "Full evaluation" if result["complete"] else "Pilot only — MAUVE incomplete"
        fig.suptitle(f"{label}: measured continuation diversity")
    return fig


def results_table(result):
    """Plain table; no plotting dependencies or unmeasured placeholders."""
    import pandas as pd

    columns = ("temperature", "amateur_window", "distinct_2", "distinct_3", "distinct_4",
               "diversity", "mauve", "perplexity", "scored_tokens", "empty_continuations", "eos_count")
    return pd.DataFrame([{key: row[key] for key in columns} for row in result["rows"]])


Writing /tmp/hw2-p2-student-9kqrawis/contrastive_utils.py


In [ ]:
import os
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import math
import importlib.util
import sys

if getattr(sys, "_hw2_p2_restart_required", False):
    raise RuntimeError("STOP: restart the runtime/session and rerun setup from the top.")
if P2_HELPER_PATH.parent != P2_WORKSPACE or not P2_HELPER_PATH.is_file() or P2_HELPER_PATH.is_symlink():
    raise RuntimeError("Run the helper export cell into this notebook's workspace first.")
# Load the exact freshly exported source, never a cwd/sibling module or stale bytecode.
_p2_spec = importlib.util.spec_from_file_location(
    f"_hw2_p2_{_P2_VARIANT}_contrastive_utils", P2_HELPER_PATH)
_p2_helper = importlib.util.module_from_spec(_p2_spec)
exec(compile(P2_HELPER_PATH.read_bytes(), str(P2_HELPER_PATH), "exec"), _p2_helper.__dict__)
sys.modules[_p2_spec.name] = _p2_helper
sys.modules["contrastive_utils"] = _p2_helper
import torch
from contrastive_utils import (
    context_limit, model_device, token_ids, environment_metadata, load_model_pair,
    load_wikitext_windows, conditional_perplexity, require_mauve, prepare_mauve,
    genuine_mauve, save_results, results_table, plot_diversity,
)

RUN_PILOT = False
RUN_FULL_EVALUATION = True
SHOW_RESULTS = True
UPSTREAM_COMMIT = "170e9142e92159c1237d731e240f5eb14aabf428"  # Set to the 40-hex SHA of your clone (graded setup task 2a); None still runs.
EXPERT_ID = "openai-community/gpt2-medium"
AMATEUR_ID = "openai-community/gpt2"
EXPERT_REVISION = "main"  # Replace with recorded resolved SHA to reproduce a previous run.
AMATEUR_REVISION = "main"
DATA_REVISION = "main"
FEATURE_REVISION = "main"
SEED = 17
FULL_SAMPLE_COUNT = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAUVE_DEVICE_ID = -1  # CPU; opt into a visible GPU index only with sufficient free VRAM.
RESULTS = None
print(environment_metadata())


{'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'packages': {'torch': '2.11.0+cpu', 'transformers': '4.57.6', 'datasets': '4.8.5', 'numpy': '1.26.4', 'mauve-text': '0.4.0', 'faiss-cpu': '1.15.1', 'scikit-learn': '1.6.1'}, 'cuda_available': False, 'cuda_devices': [], 'cpu': 'x86_64'}


### 2b. Expert–amateur implementation and ablations (3 points)
This notebook has **five** graded functions: `contrastive_scores` and `contrastive_generate`
below, then `distinct_n`, `run_experiment` and `select_examples` in Part 3.

* `contrastive_scores`: compute `log_softmax(expert_logits) - log_softmax(amateur_logits / T)`;
  set entries failing the **expert** log-space plausibility test to negative infinity. Reject
  nonfinite logits, nonpositive/nonfinite temperature, and alpha outside `(0, 1]`.
* `contrastive_generate`: select the admissible maximum at each step; PyTorch `argmax` uses the
  lowest token ID on a tie. Recompute expert and amateur forwards with **`use_cache=False`**.
  The amateur sees only the latest `amateur_window` tokens; reset learned absolute positions
  to `0..window_length-1` after truncation. This prevents hidden KV-cache leakage from dropped
  history. The expert also uses its own capacity as a rolling maximum; for the benchmark the
  768+128 tokens fit without expert truncation. Return continuation IDs **excluding terminal EOS**,
  the stop reason and EOS flag. Stop immediately on EOS or at the token budget; never pad output.
* Temperature acts on **amateur logits before normalization**, not the expert or final score.
* Test **all nine** combinations: temperatures `0.5, 1.0, 1.5` and rolling windows `1024, 512, 1`.
  The benchmark prompt is longer than 512 so the full/half contrast is meaningful at step one.

**Scope versus paper:** we implement deterministic **greedy, tokenwise CD**, not the paper's
five-beam sequence search. The handout's **rolling one-token amateur** sees only the latest token
at *every* step. This is different from the paper's restricted *initial prompt*, after which
**all generated history is retained**. Resetting positions is an additional explicit convention.
Do not call these runs an exact replication of the paper's decoding protocol.


In [ ]:
def contrastive_scores(expert_logits, amateur_logits, temperature=1.0, alpha=0.1):
    """One vocabulary vector of constrained log likelihood-ratio scores."""

    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature must be positive and finite")
    if not math.isfinite(alpha) or not 0 < alpha <= 1:
        raise ValueError("alpha must be in (0, 1]")
    if expert_logits.ndim != 1 or expert_logits.shape != amateur_logits.shape:
        raise ValueError("Expected matching 1D vocabulary vectors")

    # Temperature changes only the amateur distribution.
    expert_log_probs = torch.log_softmax(
        expert_logits.float(), dim=-1
    )
    amateur_log_probs = torch.log_softmax(
        amateur_logits.float() / temperature, dim=-1
    )

    scores = expert_log_probs - amateur_log_probs

    # In log space:
    # P_expert(token) >= alpha * max(P_expert)
    threshold = expert_log_probs.max() + math.log(alpha)
    plausible = expert_log_probs >= threshold

    return scores.masked_fill(~plausible, float("-inf"))


In [ ]:
@torch.inference_mode()
def contrastive_generate(expert, amateur, prompt_ids, *, temperature=1.0, alpha=0.1, amateur_window=1024, max_new_tokens=128, eos_token_id=None):
    """Deterministic greedy CD with fresh, rebased rolling-context forwards.

    Return schema (run_experiment and the provided helpers rely on these keys):
        {"token_ids": list[int],         # generated continuation IDs; terminal EOS excluded
         "eos_reached": bool,            # True iff generation stopped because EOS was selected
         "stop_reason": "eos" | "length"}
    """
    expert.eval()
    amateur.eval()
    if amateur_window <= 0:
        raise ValueError("amateur_window must be positive")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be nonnegative")

    vocab_size = expert.get_output_embeddings().weight.shape[0]
    context = token_ids(prompt_ids, vocab_size)
    if not context:
        raise ValueError("prompt_ids must not be empty")
    generated_ids = []
    eos_reached = False

    for _ in range(max_new_tokens):
        expert_context = context[-context_limit(expert):]
        amateur_context = context[-min(amateur_window, context_limit(amateur)):]
        expert_input = torch.tensor(
            [expert_context],
            dtype=torch.long,
            device=model_device(expert),
        )
        amateur_input = torch.tensor(
            [amateur_context],
            dtype=torch.long,
            device=model_device(amateur),
        )

        expert_positions = torch.arange(
            expert_input.shape[1], device=expert_input.device
        ).unsqueeze(0)
        amateur_positions = torch.arange(
            amateur_input.shape[1], device=amateur_input.device
        ).unsqueeze(0)

        expert_logits = expert(
            input_ids=expert_input,
            position_ids=expert_positions,
            use_cache=False,
        ).logits[0, -1]

        amateur_logits = amateur(
            input_ids=amateur_input,
            position_ids=amateur_positions,
            use_cache=False,
        ).logits[0, -1]

        scores = contrastive_scores(
            expert_logits,
            amateur_logits.to(expert_logits.device),
            temperature=temperature,
            alpha=alpha,
        )
        next_token = int(scores.argmax().item())

        if eos_token_id is not None and next_token == eos_token_id:
            eos_reached = True
            break

        generated_ids.append(next_token)
        context.append(next_token)

    return {
        "token_ids": generated_ids,
        "eos_reached": eos_reached,
        "stop_reason": "eos" if eos_reached else "length",
    }


## Part 3 — Evaluation and analysis (14 points)
### 3a. Nine-condition generation and metrics (8 points)
**Predeclare the measurements before generating anything.**

* **Diversity:** continuation-only corpus-level distinct-2, distinct-3, distinct-4. Split each
  decoded continuation on whitespace, retaining case and punctuation; never join n-grams across
  samples. Numerator = unique n-grams over the corpus; denominator = total within-sample n-gram
  occurrences over the corpus. An empty denominator gives 0. Longer samples affect this micro
  aggregation more. It is not the paper's repetition-based diversity product; we use the handout's
  allowed distinct-n option. **Ranking** is the arithmetic mean of these three distinct-n values,
  decreasing; break exact ties by increasing temperature, then decreasing window size. Best =
  first and worst = last in that total ordering. Ties are not evidence of a substantive difference.
* **MAUVE:** genuine `mauve.compute_mauve`, human continuations as `p_text`, generated continuations
  as `q_text`, no prompts. Fixed GPT-2 feature-model revision, `max_text_length=128`, 25 buckets,
  5 k-means redos, 500 k-means iterations, scaling factor 5, seed 17, batch size 8. Report these
  settings: scores change with feature extractor, sample size and clustering. Empty texts or missing
  dependencies make the full evaluation incomplete, not a reason to silently filter examples.
  No proxy or dummy MAUVE numbers are accepted. Feature tokens differ from whitespace diversity tokens.
* **Coherence proxy:** conditional continuation perplexity under the **same fixed expert scorer**
  for every condition. Sum autoregressive negative log likelihood on generated text tokens only;
  divide by the total scored continuation tokens, then exponentiate. Prompt/padding labels are `-100`.
  This is NOT mean per-sequence perplexity and is NOT direct semantic coherence or the paper's SimCSE
  metric. Using the expert/GPT-2 family biases evaluation toward its own style and probabilities.
  Document that dependence rather than interpreting low perplexity as factual correctness.
* **EOS:** terminate and omit EOS from decoded text, diversity, MAUVE and NLL. Thus generated lengths
  may be shorter than the 128-token human continuation. Record token counts, empty outputs and EOS
  counts; discuss this length confound. Empty continuations contribute no NLL targets; all-empty
  results yield `None`, not a fabricated perplexity. Full MAUVE then fails explicitly.

MAUVE is deliberately unavailable in the small pilot; its rows carry `None` and an incomplete
status. Results tables and plots are created only from actual returned measurements. A full run
is expensive: two uncached forwards per generated token for 9 × sample-count × 128 steps, plus
scoring and MAUVE features. Use a GPU and budget accordingly. Never label the pilot a full run.


In [ ]:
def distinct_n(texts, n):
    """Corpus micro distinct-n; whitespace tokens, no cross-sample n-grams."""
    unique_ngrams = set()
    total_ngrams = 0

    for text in texts:
        words = text.split()

        for i in range(len(words) - n + 1):
            ngram = tuple(words[i:i + n])
            unique_ngrams.add(ngram)
            total_ngrams += 1

    if total_ngrams == 0:
        return 0.0

    return len(unique_ngrams) / total_ngrams


In [ ]:
def run_experiment(expert, amateur, tokenizer, windows, *, alpha=0.1, max_new_tokens=128, full_run=False, mauve_settings=None, provenance=None, checkpoint_dir=None):
    """Run exactly nine matched conditions; incomplete pilots are labeled explicitly.

    Return schema (results_table, plot_diversity, save_results and the display cell read these keys;
    everything must be JSON-serializable, since save_results uses allow_nan=False):
        {"complete": bool,               # True only for a full run with genuine MAUVE
         "status": "full_evaluation" | "pilot_incomplete_no_mauve",
         "rows": [row, ...],             # exactly nine: temperatures 0.5/1.0/1.5 x windows 1024/512/1
         "windows": windows,             # the matched input windows, unchanged and in source order
         "provenance": dict,             # the provenance argument (or {})
         "environment": dict,            # environment_metadata()
         "mauve_settings": dict | None,
         "parameters": dict}             # alpha, max_new_tokens, grid, algorithm/EOS/ranking notes
    Each row:
        {"temperature": float, "amateur_window": int,
         "distinct_2": float, "distinct_3": float, "distinct_4": float,
         "diversity": float,             # mean of distinct_2/3/4
         "mauve": float | None,          # None in the pilot (no MAUVE)
         "nll_sum": float, "scored_tokens": int, "mean_nll": float | None,
         "perplexity": float | None, "empty_continuations": int,  # all from conditional_perplexity
         "eos_count": int,               # generations with eos_reached
         "generations": [               # one per window, same order as windows
             {"window_id": str, "text": str,  # text = decoded token_ids
              "token_ids": list[int], "eos_reached": bool, "stop_reason": str}, ...]}
    """
    temperatures = (0.5, 1.0, 1.5)
    amateur_windows = (1024, 512, 1)

    if not windows:
        raise ValueError("No input windows provided")

    if full_run:
        if mauve_settings is None:
            raise ValueError("Full evaluation requires MAUVE settings")
        require_mauve()

    # Checkpoints are isolated by mode and validated before any reuse.
    import json
    from pathlib import Path
    checkpoint_root = Path(checkpoint_dir if checkpoint_dir is not None else P2_RESULTS_DIR)
    checkpoint_root = checkpoint_root / ("full_checkpoints" if full_run else "pilot_checkpoints")
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    signature = json.loads(json.dumps({
        "windows": windows, "alpha": alpha, "max_new_tokens": max_new_tokens,
        "full_run": full_run, "mauve_settings": mauve_settings,
        "provenance": provenance if provenance is not None else {},
        "tokenizer": {"eos_token_id": tokenizer.eos_token_id,
                      "pad_token_id": tokenizer.pad_token_id},
    }, allow_nan=False))
    print("Checkpoint directory:", checkpoint_root.resolve(), flush=True)

    prompts = [window["prompt_ids"] for window in windows]
    references = [window["reference_text"] for window in windows]
    rows = []

    for temperature in temperatures:
        for amateur_window in amateur_windows:
            print(
                f"Temperature={temperature}, "
                f"window={amateur_window}",
                flush=True,
            )

            checkpoint = checkpoint_root / f"generation_T{temperature}_W{amateur_window}.json"
            saved = {"signature": signature, "temperature": temperature,
                     "amateur_window": amateur_window, "generations": []}
            if checkpoint.exists():
                saved = json.loads(checkpoint.read_text(encoding="utf-8"))
                if (saved.get("signature") != signature
                        or saved.get("temperature") != temperature
                        or saved.get("amateur_window") != amateur_window):
                    raise ValueError(f"Checkpoint settings/inputs changed: {checkpoint}. "
                                     "Use a new results directory for a different experiment.")
            generations = saved["generations"]
            if (len(generations) > len(windows)
                    or [g["window_id"] for g in generations]
                    != [w["window_id"] for w in windows[:len(generations)]]):
                raise ValueError(f"Checkpoint sample order is invalid: {checkpoint}")
            if "row" in saved:
                if len(generations) != len(windows):
                    raise ValueError(f"Completed checkpoint is incomplete: {checkpoint}")
                rows.append(saved["row"])
                print("  Reusing completed condition", flush=True)
                continue
            print(f"  Resuming from {len(generations)}/{len(windows)}", flush=True)

            for index in range(len(generations), len(windows)):
                window = windows[index]
                output = contrastive_generate(
                    expert,
                    amateur,
                    window["prompt_ids"],
                    temperature=temperature,
                    alpha=alpha,
                    amateur_window=amateur_window,
                    max_new_tokens=max_new_tokens,
                    eos_token_id=tokenizer.eos_token_id,
                )

                text = tokenizer.decode(
                    output["token_ids"],
                    skip_special_tokens=False,
                    clean_up_tokenization_spaces=False,
                )

                generations.append({
                    "window_id": window["window_id"],
                    "text": text,
                    "token_ids": output["token_ids"],
                    "eos_reached": output["eos_reached"],
                    "stop_reason": output["stop_reason"],
                })

                # save_results writes a temporary file then atomically replaces the JSON.
                # Save BEFORE metrics, so a metric failure never discards generation work.
                save_results(checkpoint, saved)
                print(
                    f"  Generated {index + 1}/{len(windows)}",
                    flush=True,
                )

            texts = [item["text"] for item in generations]
            continuations = [
                item["token_ids"] for item in generations
            ]

            distinct_2 = distinct_n(texts, 2)
            distinct_3 = distinct_n(texts, 3)
            distinct_4 = distinct_n(texts, 4)

            print("  Computing perplexity...", flush=True)
            coherence = conditional_perplexity(
                expert,
                prompts,
                continuations,
                pad_token_id=(
                    tokenizer.pad_token_id
                    if tokenizer.pad_token_id is not None
                    else tokenizer.eos_token_id
                ),
                batch_size=1,
            )

            mauve_score = None
            if full_run:
                print("  Computing MAUVE...", flush=True)
                mauve_score = genuine_mauve(
                    references, texts, mauve_settings
                )

            row = {
                "temperature": temperature,
                "amateur_window": amateur_window,
                "distinct_2": distinct_2,
                "distinct_3": distinct_3,
                "distinct_4": distinct_4,
                "diversity": (
                    distinct_2 + distinct_3 + distinct_4
                ) / 3,
                "mauve": mauve_score,
                **coherence,
                "eos_count": sum(
                    item["eos_reached"] for item in generations
                ),
                "generations": generations,
            }
            saved["row"] = row
            save_results(checkpoint, saved)
            rows.append(row)

    return {
        "complete": bool(full_run),
        "status": (
            "full_evaluation"
            if full_run
            else "pilot_incomplete_no_mauve"
        ),
        "rows": rows,
        "windows": windows,
        "provenance": provenance if provenance is not None else {},
        "environment": environment_metadata(),
        "mauve_settings": mauve_settings if full_run else None,
        "parameters": {
            "alpha": alpha,
            "max_new_tokens": max_new_tokens,
            "temperatures": list(temperatures),
            "amateur_windows": list(amateur_windows),
            "algorithm": (
                "Greedy expert-amateur contrastive decoding; "
                "temperature applies only to amateur; "
                "uncached rolling contexts with rebased positions"
            ),
            "eos_policy": (
                "Stop on EOS; exclude terminal EOS from "
                "continuation IDs and all metrics"
            ),
            "ranking": (
                "Mean distinct-2/3/4 descending; ties broken by "
                "temperature ascending, then window descending"
            ),
        },
    }


In [ ]:
def select_examples(result, count=3):
    """Return real matched prompts and raw best/worst continuations, never templates.

    Return schema (read by the display cell):
        {"best":  {"temperature": float, "amateur_window": int, "diversity": float},
         "worst": {"temperature": float, "amateur_window": int, "diversity": float},
         "all_diversity_tied": bool,     # best and worst diversity are exactly equal
         "examples": [                   # the first `count` windows of result["windows"]
             {"window_id": str,
              "prompt": str,             # window["prompt_text"]
              "human": str,              # window["reference_text"]
              "best": str,               # best setting's generation text for this window
              "worst": str}, ...]}       # worst setting's generation text for this window
    """
    if not result["rows"]:
      raise ValueError("No experiment results available")
    if not isinstance(count, int) or isinstance(count, bool) or count < 1:
        raise ValueError("count must be a positive integer")
    ranked = sorted(
        result["rows"],
        key=lambda row: (
            -row["diversity"],
            row["temperature"],
            -row["amateur_window"],
        ),
    )

    best_row = ranked[0]
    worst_row = ranked[-1]
    best_texts = {
        item["window_id"]: item["text"]
        for item in best_row["generations"]
    }
    worst_texts = {
        item["window_id"]: item["text"]
        for item in worst_row["generations"]
    }

    examples = []
    for window in result["windows"][:count]:
        window_id = window["window_id"]
        examples.append({
            "window_id": window_id,
            "prompt": window["prompt_text"],
            "human": window["reference_text"],
            "best": best_texts[window_id],
            "worst": worst_texts[window_id],
        })

    keys = ("temperature", "amateur_window", "diversity")

    return {
        "best": {key: best_row[key] for key in keys},
        "worst": {key: worst_row[key] for key in keys},
        "all_diversity_tied": (
            best_row["diversity"] == worst_row["diversity"]
        ),
        "examples": examples,
    }


In [ ]:
if getattr(sys, "_hw2_p2_restart_required", False):
    raise RuntimeError("STOP: restart the runtime/session and rerun setup before experiments.")
if RUN_PILOT and RUN_FULL_EVALUATION:
    raise ValueError("Choose either the pilot or the full evaluation, not both")
if RUN_PILOT or RUN_FULL_EVALUATION:
    if UPSTREAM_COMMIT is not None and (not isinstance(UPSTREAM_COMMIT, str) or
            len(UPSTREAM_COMMIT) != 40 or any(c not in "0123456789abcdefABCDEF" for c in UPSTREAM_COMMIT)):
        raise ValueError("UPSTREAM_COMMIT must be None or an actual 40-character hexadecimal checkout SHA")
    if RUN_FULL_EVALUATION:
        require_mauve()  # Preflight BEFORE downloading expert/data.
        if (isinstance(MAUVE_DEVICE_ID, bool) or not isinstance(MAUVE_DEVICE_ID, int) or
                MAUVE_DEVICE_ID < -1 or MAUVE_DEVICE_ID >= torch.cuda.device_count()):
            raise ValueError("MAUVE_DEVICE_ID must be -1 (CPU) or an available CUDA device index")
        print("Full 100-window evaluation takes roughly 7-8 hours on a T4 and may exceed Colab memory or session time; save results promptly.")
    torch.manual_seed(SEED)
    expert, amateur, tokenizer, model_metadata = load_model_pair(
        EXPERT_ID, AMATEUR_ID, EXPERT_REVISION, AMATEUR_REVISION, DEVICE)
    windows, data_metadata = load_wikitext_windows(tokenizer,
        count=FULL_SAMPLE_COUNT if RUN_FULL_EVALUATION else 3,
        prompt_tokens=768 if RUN_FULL_EVALUATION else 64,
        reference_tokens=128 if RUN_FULL_EVALUATION else 32,
        seed=SEED, revision=DATA_REVISION)
    mauve_settings = (prepare_mauve(seed=SEED, revision=FEATURE_REVISION,
        device_id=MAUVE_DEVICE_ID) if RUN_FULL_EVALUATION else None)
    # Mount Google Drive first for persistence across Colab runtime deletion:
    # from google.colab import drive
    # drive.mount("/content/drive")
    # You can set EVALUATION_RESULTS_DIR to another persistent directory before this cell.
    _drive_root = Path("/content/drive/MyDrive")
    P2_RESULTS_DIR = Path(globals().get("EVALUATION_RESULTS_DIR",
        (_drive_root / "HW2Problem2_results" if _drive_root.is_dir()
         else Path.cwd() / "HW2Problem2_results")))
    P2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print("Results and resume files:", P2_RESULTS_DIR.resolve(), flush=True)
    if not _drive_root.is_dir():
        print("WARNING: local checkpoints survive reruns, but runtime deletion can erase them. "
              "For Colab, mount Google Drive before this cell or use a persistent "
              "EVALUATION_RESULTS_DIR.", flush=True)
    RESULTS = run_experiment(expert, amateur, tokenizer, windows,
        max_new_tokens=128 if RUN_FULL_EVALUATION else 32, full_run=RUN_FULL_EVALUATION,
        mauve_settings=mauve_settings, checkpoint_dir=P2_RESULTS_DIR,
        provenance={"models": model_metadata, "dataset": data_metadata, "seed": SEED,
                    "reference_repository": "https://github.com/XiangLi1999/ContrastiveDecoding",
                    "reference_commit": UPSTREAM_COMMIT, "implementation": "independent; no upstream code copied"})
    save_results(P2_RESULTS_DIR / ("problem2_full.json" if RUN_FULL_EVALUATION else "problem2_pilot.json"), RESULTS)


In [ ]:
# Automatic checkpoint saving/resume is now handled by run_experiment above.
# Rerun the evaluation cell with the same results directory to continue.
# The old manual rescue loop is no longer needed.


In [ ]:
if SHOW_RESULTS:
    if RESULTS is None:
        raise RuntimeError("No measured results: run a pilot/full evaluation first")
    from IPython.display import display
    display(results_table(RESULTS))
    fig = plot_diversity(RESULTS)
    display(fig)
    import matplotlib.pyplot as plt
    plt.close(fig)  # Prevent the inline backend from showing the figure a second time.
    examples = select_examples(RESULTS, count=3)
    print("Measured best:", examples["best"], "Measured worst:", examples["worst"])
    if examples["all_diversity_tied"]:
        print("All diversity scores tie; ordering is only the predeclared tie-break.")
    for example in examples["examples"]:
        print("\nWINDOW", example["window_id"])
        print("PROMPT (raw):\n", example["prompt"])
        print("HUMAN CONTINUATION (raw):\n", example["human"])
        print("BEST CONTINUATION (raw):\n", example["best"])
        print("WORST CONTINUATION (raw):\n", example["worst"])


NameError: name 'SHOW_RESULTS' is not defined

In [ ]:
# Runtime storage is ephemeral. Opt in to download only your measured results.
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    if not P2_RESULTS_DIR.is_dir() or not any(P2_RESULTS_DIR.iterdir()):
        raise RuntimeError("No saved results yet; run an experiment first.")
    archive = shutil.make_archive(str(P2_WORKSPACE / "problem2-results"), "zip", P2_RESULTS_DIR)
    print("Results archive:", archive)
    try:
        from google.colab import files
    except ImportError:
        print("Local runtime: save the archive from the path above.")
    else:
        files.download(archive)


### 3b. Analysis and relation to the paper (3 points)
**Pending actual full evaluation — do not infer winners from the algorithm or offline unit tests.**
Report the measured top setting for each of diversity (higher), MAUVE (higher) and conditional
perplexity (lower), including exact ties. Contrast the 1024/512/1 context curves across temperatures.
Describe metric disagreements, length/EOS effects, uncertainty from finite samples, and scorer-family
bias. Do not interpret diversity alone as quality. How do the findings compare with the paper,
considering greedy versus five-beam search, rolling versus prompt-only context restriction,
model sizes, sampling windows, distinct-n versus repetition diversity, and perplexity versus SimCSE?

**Your measured discussion:** TODO after running the full evaluation. No empirical answer is supplied.

### 3c. Qualitative examples (3 points)
Use `select_examples` to display **three identical prompts**, their human continuations, and the
raw generated continuations under the measured best/worst diversity settings. They are the first
three selected windows in source order, not cherry-picked or invented passages. The selector uses
the declared total ranking; if all scores tie, explicitly report that best/worst labels are arbitrary.
For each pair, comment on repetition, topic drift, specificity and grammatical continuity using
quotations from those actual continuations; distinguish observations from metric interpretations.

**Your comments on actual examples:** TODO after results exist.

## Reproducibility and submission checklist
- [ ] Answer all three 2-point conceptual questions and complete the notebook's graded functions.
- [ ] Record actual reference checkout SHA, stock-library versions, model/data/feature SHAs,
      dataset fingerprint, seeds, hardware and setup steps. Do not install the reference fork.
- [ ] Verify compatible tokenizer pipeline and same-family smaller amateur.
- [ ] Submit this notebook (or a Markdown file plus your code files, per the HW2 handout), including the embedded helper export.
- [ ] Run the 768+128 protocol on exactly nine settings, saving raw outputs/IDs and provenance.
- [ ] Run the optional MAUVE install cell for full evaluation; report missing metrics as incomplete.
- [ ] Include the genuine nine-row table, measured diversity figure, metric discussion and 2–3
      matched best/worst examples. A pilot is not a substitute.
- [ ] Explain EOS/length effects, conditional-NLL denominator, metric limitations, greedy beam-search
      difference and rolling-context difference from the paper.

### References and provenance
- Li et al., *Contrastive Decoding: Open-ended Text Generation as Optimization*:
  [paper](https://arxiv.org/abs/2210.15097), especially objective, Section 3.4 and Section 5.1.
- [Original repository](https://github.com/XiangLi1999/ContrastiveDecoding), reference checkout only.
  This notebook does not copy its implementation or assume a repository-wide license.
- [MAUVE](https://github.com/krishnap25/mauve): genuine distributional comparison, not a handmade proxy.
